# Fix Embedded Documents with Incorrect Metadata

This notebook fixes documents that were embedded without proper metadata initialization.
It will:
1. Query Qdrant for documents with missing metadata (fincode=None, ticker=None)
2. Look up correct metadata using attachment filename
3. Rebuild page_content with correct prefix
4. Delete old embeddings
5. Re-embed with corrected data

In [ ]:
import asyncio
import logging
import re
from collections import defaultdict

from dotenv import load_dotenv
from langchain_core.documents import Document

from rag.ingestion.vector_store import QdrantManager
from utils.data_helpers import (
    fincode_to_symbol,
    get_metadata_item_by_attachment_name,
    initialize_metadata_data,
    initialize_stock_data,
)

load_dotenv()

logger = logging.getLogger(__name__)

# Initialize data
await initialize_stock_data()
await initialize_metadata_data()

qdrant_manager = QdrantManager()

## Step 1: Query Documents with Missing Metadata

In [ ]:
async def get_documents_with_missing_metadata():
    """Query Qdrant for all documents with missing metadata."""
    client = await qdrant_manager._ensure_client()
    
    logger.info("Scrolling through all points to find documents with missing metadata...")
    
    # Scroll through all points
    offset = None
    all_points = []
    
    while True:
        scroll_result = await asyncio.to_thread(
            client.scroll,
            collection_name=qdrant_manager.collection_name,
            limit=100,
            offset=offset,
            with_payload=True,
            with_vectors=False,
        )
        
        points, next_offset = scroll_result
        all_points.extend(points)
        
        if next_offset is None:
            break
        offset = next_offset
    
    logger.info(f"Found {len(all_points)} total points in collection")
    
    # Filter points with missing metadata
    docs_with_missing_metadata = []
    
    for point in all_points:
        if point.payload:
            metadata = point.payload.get("metadata", {})
            page_content = point.payload.get("page_content", "")
            
            # Check if metadata is missing or null
            has_missing = (
                metadata.get("fincode") is None or
                metadata.get("ticker") is None or
                metadata.get("category") == "" or
                metadata.get("document_date") == ""
            )
            
            if has_missing:
                docs_with_missing_metadata.append({
                    "point_id": point.id,
                    "metadata": metadata,
                    "page_content": page_content,
                })
    
    logger.info(f"Found {len(docs_with_missing_metadata)} documents with missing metadata")
    
    return docs_with_missing_metadata

docs_to_fix = await get_documents_with_missing_metadata()

# Group by source
docs_by_source = defaultdict(list)
for doc in docs_to_fix:
    source = doc["metadata"].get("source", "unknown")
    docs_by_source[source].append(doc)

logger.info(f"Grouped into {len(docs_by_source)} unique sources")
for source, docs in list(docs_by_source.items())[:5]:
    logger.info(f"  {source}: {len(docs)} chunks")

## Step 2: Fix Metadata and Page Content

In [ ]:
def fix_document(doc_data: dict) -> Document:
    """Fix a single document's metadata and page_content prefix."""
    metadata = doc_data["metadata"].copy()
    page_content = doc_data["page_content"]
    source = metadata.get("source", "")
    
    # Look up correct metadata
    file_meta = get_metadata_item_by_attachment_name(source) or {}
    
    # Extract fincode, category, document_date from file metadata
    fincode = file_meta.get("fincode", None)
    category = file_meta.get("subcatname", "")
    document_date = file_meta.get("newsDt", "")[:10] if file_meta.get("newsDt") else ""
    
    # Get ticker from fincode
    ticker = None
    if fincode:
        ticker = fincode_to_symbol(fincode)
    
    # Extract page number from old prefix
    page_no_match = re.search(r"page no: (\d+)", page_content)
    page_no = page_no_match.group(1) if page_no_match else ""
    
    # Remove old prefix (everything before first newline)
    content_parts = page_content.split("\n", 1)
    actual_content = content_parts[1] if len(content_parts) > 1 else page_content
    
    # Build new prefix
    new_prefix = f"Ticker: {ticker}; Type: {category}; Date: {document_date}; fincode: {fincode}; page no: {page_no}\n"
    new_page_content = new_prefix + actual_content
    
    # Update metadata
    metadata["fincode"] = fincode
    metadata["ticker"] = ticker
    metadata["category"] = category
    metadata["document_date"] = document_date
    
    return Document(page_content=new_page_content, metadata=metadata)

# Test with first document
if docs_to_fix:
    test_doc = fix_document(docs_to_fix[0])
    logger.info("\nTest fixed document:")
    logger.info(f"Source: {test_doc.metadata.get('source')}")
    logger.info(f"Fincode: {test_doc.metadata.get('fincode')}")
    logger.info(f"Ticker: {test_doc.metadata.get('ticker')}")
    logger.info(f"Category: {test_doc.metadata.get('category')}")
    logger.info(f"Date: {test_doc.metadata.get('document_date')}")
    logger.info(f"Page content preview: {test_doc.page_content[:200]}...")

## Step 3: Delete and Re-embed Documents by Source

**WARNING:** This will delete and re-embed documents. Make sure you have a backup!

In [ ]:
# Set to True to actually perform deletion and re-embedding
DRY_RUN = True

if DRY_RUN:
    logger.warning("DRY RUN MODE - No changes will be made")
else:
    logger.warning("LIVE MODE - Documents will be deleted and re-embedded!")

async def fix_source_documents(source: str, doc_list: list[dict]):
    """Fix all documents from a single source."""
    logger.info(f"\nProcessing source: {source} ({len(doc_list)} chunks)")
    
    # Fix all documents
    fixed_docs = [fix_document(doc_data) for doc_data in doc_list]
    
    # Log sample
    sample = fixed_docs[0]
    logger.info(f"  Fincode: {sample.metadata.get('fincode')}")
    logger.info(f"  Ticker: {sample.metadata.get('ticker')}")
    logger.info(f"  Category: {sample.metadata.get('category')}")
    logger.info(f"  Date: {sample.metadata.get('document_date')}")
    
    if not DRY_RUN:
        # Delete old embeddings
        logger.info(f"  Deleting old embeddings for {source}...")
        delete_result = await qdrant_manager.delete_by_source(source)
        logger.info(f"  Deleted {delete_result['deleted_count']} points")
        
        # Re-embed with correct data
        logger.info(f"  Re-embedding {len(fixed_docs)} documents...")
        embed_result = await qdrant_manager.embed_documents(fixed_docs, show_progress=False)
        logger.info(
            f"  Embedded {embed_result['total_embedded']} documents "
            f"(Cost: ${embed_result['estimated_cost_usd']:.4f})"
        )
    else:
        logger.info("  [DRY RUN] Would delete and re-embed")
    
    return len(fixed_docs)

# Process first 5 sources (or all if you set DRY_RUN=False)
total_fixed = 0
sources_to_process = list(docs_by_source.items())[:5]  # Limit for testing

for source, doc_list in sources_to_process:
    try:
        fixed_count = await fix_source_documents(source, doc_list)
        total_fixed += fixed_count
    except Exception as e:
        logger.error(f"Error processing {source}: {e}")
        import traceback
        logger.error(traceback.format_exc())

logger.info(f"\nTotal: Fixed {total_fixed} documents from {len(sources_to_process)} sources")

if DRY_RUN:
    logger.warning(
        "\nDRY RUN completed. Set DRY_RUN=False to actually fix documents.\n"
        f"Total sources with issues: {len(docs_by_source)}"
    )

## Step 4: Process All Sources (After Testing)

Once you've verified the dry run works correctly, use this cell to process all sources.

In [ ]:
# Uncomment and run this cell to process ALL sources
# Make sure to set DRY_RUN=False in the cell above first!

# total_fixed = 0
# total_cost = 0.0

# for idx, (source, doc_list) in enumerate(docs_by_source.items(), 1):
#     try:
#         logger.info(f"\n[{idx}/{len(docs_by_source)}] Processing {source}...")
#         fixed_count = await fix_source_documents(source, doc_list)
#         total_fixed += fixed_count
#     except Exception as e:
#         logger.error(f"Error processing {source}: {e}")
#         import traceback
#         logger.error(traceback.format_exc())
#         continue

# logger.info(f"\n=== COMPLETE ===")
# logger.info(f"Total documents fixed: {total_fixed}")
# logger.info(f"Total sources fixed: {len(docs_by_source)}")

## Step 5: Verify Fix

Query Qdrant again to verify documents now have correct metadata.

In [ ]:
# Re-check for documents with missing metadata
docs_still_missing = await get_documents_with_missing_metadata()

logger.info(f"\nDocuments still with missing metadata: {len(docs_still_missing)}")

if docs_still_missing:
    logger.warning("Some documents still have missing metadata. Review:")
    for doc in docs_still_missing[:3]:
        logger.info(f"  Source: {doc['metadata'].get('source')}")
        logger.info(f"  Fincode: {doc['metadata'].get('fincode')}")
        logger.info(f"  Ticker: {doc['metadata'].get('ticker')}")
else:
    logger.info("✓ All documents now have complete metadata!")